# Part 3: Agent-driven policy research

In Parts 1 and 2, you saw that serving agentic workloads involves competing system tradeoffs: concurrency vs. memory pressure, prefix-tree eviction vs. recency, job-level vs. call-level dispatch, and cache-affinity routing vs. load balancing across replicas. A static policy tuned for one workload family often regresses on another.

In this part, **you will design and implement your own agentic research system in `research/`** that uses LLM agents to iteratively propose hypotheses, write candidate serving policies, evaluate them on the simulator, and select a final policy. Beyond achieving a strong speedup score, **your primary goal is to understand both the research process and the resulting policy**:

- **How your research system behaves:** Which architectural choices, diagnostic feedback, and search objectives actually help agents write valid, higher-performing policies, and where do agents get stuck or waste tokens?
- **What policy your agents discovered and why it works:** What scheduling, routing, concurrency, and eviction mechanisms did your agents invent? Which mechanisms genuinely reduce job latency (verified by ablating them), and on which workloads does the policy still fail?

![One possible research system and the evaluation splits.](../docs/figures/research_system.svg)

A **campaign** is one unattended run of your research system from a declared starting point, configuration, and budget to final candidate selection. While you may build, debug, and refine your research system across runs, your submitted policy must be generated and selected during an unattended campaign without manual code edits. See the [Part 3 handout](../docs/part3_handout.md) for the full rules on autonomy, intervention logs, and sandbox constraints.

Use this notebook to inspect workloads, test your evaluator, and analyze campaign logs. Keep your research system and generated policies in separate files so campaigns can run unattended from the command line:

```text
research/      your agentic research system, evaluator, configuration, and prompts
candidates/    all generated candidate policies, including failed attempts
runs/          campaign logs with lineage, scores, and token usage
```

**Grading (70 points):** policy performance on the hidden test set (50 points); research report answering Q1–Q6 (20 points). Both team members must also be ready to explain how your research system and final policy work during the individual check.

## Evaluator setup

Part 3 evaluates policies on **two independent replicas** of Llama-3-8B (each on a simulated NVIDIA A100 80 GB GPU with its own request queue and 16-token KV cache). A call can reuse a cached prefix only if `route_call` sends it to a replica where that prefix is currently resident, creating a tradeoff between cache affinity and load balancing.

`score(..., backend="sim")` uses this two-replica configuration and starts every trace with empty caches and fresh policy state. See [Evaluator configuration](../README.md#evaluator-configuration) for details.

## Before you begin

Complete the [environment setup](../README.md#getting-started) before running this notebook, and select **CS 378 Assignment 4** as the notebook kernel.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

from agentkv import viz
from agentkv.notebook import check, repo_root

os.chdir(repo_root())          # every path below is relative to the repository root
plt.rcParams["figure.dpi"] = 100

from agentkv import score
from agentkv.trace import load

DEV = sorted(Path("data/traces/public_dev").glob("*.json"))
VAL = sorted(Path("data/traces/public_val").glob("*.json"))
print(f"{len(DEV)} dev traces, {len(VAL)} validation traces")

## 1. Performance metric and baseline

Policies are scored by how much they reduce mean **job latency** (the time from a job's arrival until its final LLM or tool operation completes) relative to `policies/reference.py` under the exact same workload and two-replica configuration.

The overall speedup **S** weights all eight workload families equally (taking the geometric mean across traces within each family, and then across the eight families), and `g = 1 − 1/S` is the corresponding fractional reduction in job latency. Performance points are computed as:

```text
performance points = clip(25 + 25 × g_eff / 0.10, 0, 50)
```

where `g_eff = g`, except that if any family's mean job latency is more than 20% worse than the baseline's (`by_family[f] < 1 / 1.2 ≈ 0.8333`), `g_eff` is capped at `0` (capping points at 25/50 even if overall `S > 1`). Invalid or incomplete evaluations receive 0 points.

| Speedup S | 0.95 | 1.00 (baseline) | 1.02 | 1.05 | 1.08 | 1.112 or more |
|---|---|---|---|---|---|---|
| Performance points (assuming no >20% family regression) | 11.8 | 25.0 | 29.9 | 36.9 | 43.5 | 50.0 |

Pass your completed Part 2 `policies/reference.py` (which uses `cap = 8`, least-loaded routing, a 2048-token batch budget, FCFS dispatch/admission, and LRU eviction) and `agentkv/policy.py` to your research agents as their starting point, and save generated policies in `candidates/`. (Official speedups are always computed against fixed staff reference results; `policies/fifo.py` is also available as a FIFO comparison.)

**Important boundary:** Your outer research system in `research/` may inspect family labels (`D`, `W1–W5`, `M1–M2`) and per-trace diagnostics to analyze failures and plan experiments, but a policy running inside `score()` cannot see trace filenames or family labels—any runtime adaptation inside `policy.py` must rely solely on causal events (`on_event`) and `View` state.

In [ ]:
reference = score("policies/reference.py", split="public_dev")
assert reference.valid, reference.violations
print(reference.summary())  # the reference should score 1.0 against itself

## 2. Understand the eight workload families

Before building your research system, inspect what makes the eight workload families different (four traces per family in `public_dev`, two in `public_val`):

- **D (Deep research):** Fixed staff traces of the Part 1 agent under four conditions (high reuse, tight memory, topic/corpus shift, and bursty arrivals with slow tools).
- **W1–W5 (Real-world agent & chat traces):** Mooncake long-context chat (`W1`), SWE-bench coding agents (`W2`), LooGLE document QA (`W3`), τ²-bench customer-service agents (`W4`), and Alibaba Qwen Bailian reasoning traces (`W5`).
- **M1–M2 (Shared-engine mixtures):** Document QA mixed with long-context chat (`M1`) or long-output reasoning (`M2`) on the same serving engine.

![The eight workload families used for evaluation.](../docs/figures/families.svg)

Run the cell below to compare call counts per job, prompt and output lengths, potential prefix sharing, and per-replica KV-cache capacity across `public_dev`. Notice why no single static concurrency cap or naive eviction rule is optimal everywhere.

In [ ]:
def summarize(path):
    t = load(path)
    calls = [n for _, n in t.llm_nodes()]
    seen, shared, total = set(), 0, 0
    for job in sorted(t.jobs, key=lambda j: j.arrival):
        for n in (x for x in job.nodes if x.type == "llm"):
            full = n.hashes[: n.n_prompt // t.memory.block_size]
            shared += sum(h in seen for h in full) * t.memory.block_size
            total += n.n_prompt
            seen.update(full)
    return {"trace": t.trace_id, "family": t.family, "calls/job": len(calls) / len(t.jobs),
            "prompt tok": sum(n.n_prompt for n in calls) / len(calls),
            "output tok": sum(n.n_output for n in calls) / len(calls),
            "shared prefix": shared / total, "memory (blocks)": t.memory.num_blocks,
            "description": t.description}

rows = [summarize(p) for p in DEV]
print(f"{'trace':10} {'calls/job':>9} {'prompt tok':>10} {'output tok':>10} {'shared':>7} {'memory':>7}")
for r in rows:
    print(f"{r['trace']:10} {r['calls/job']:9.1f} {r['prompt tok']:10.0f} {r['output tok']:10.0f} "
          f"{r['shared prefix']:7.0%} {r['memory (blocks)']:7d}")
print()
for r in rows[::4]:                  # one trace per family; D's four traces are four different conditions
    print(f"{r['trace']}: {r['description']}")

You can replay any trace on two replicas using `viz.record(..., replicas=2)` to see where the reference policy spends time waiting for concurrency slots, waiting for replica admission, or evicting reusable prefixes. You will also use this visualization in **Q5** and **Q6** to explain how your final policy behaves on specific traces.

In [ ]:
trace, policy, params = "data/traces/public_dev/dev-W4-0.json", "policies/reference.py", {}
rec = viz.record(trace, policy, params=params, replicas=2)
fig, axes = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={"height_ratios": [2.4, 1]})
viz.timeline(rec, axes[0])
viz.memory(rec, axes[1])
fig.tight_layout()
plt.show()

## 3. Design the evaluator for your research system (`research/evaluator.py`)

`score()` runs a candidate policy in the sandbox and returns overall speedup (`r.speedup`), per-family speedups (`r.by_family`), validity/violations (`r.valid`, `r.violations`), and detailed per-trace diagnostics (`r.by_trace`, including cache hit rates, recomputed tokens, preemptions, busy/prefill time, and hook CPU time).

In [ ]:
r = score("policies/reference.py", split="public_dev", params={"cap": 8})
print(r.summary())
viz.families(r)
plt.show()

To guide your agents effectively, implement an evaluator in `research/evaluator.py` (starting from `examples/evaluator.py`) that calls `score()` and decides:

1. **What `objective` to maximize (`None` if `not result.valid`):** You might optimize overall `result.speedup`, penalize any family that approaches the 20% regression cliff, focus a sub-experiment on a weak family (e.g., `result.by_family["W3"]`), or maximize worst-family speedup (`min(result.by_family.values())`).
2. **What diagnostic feedback to show your agents:** If an agent only sees `S = 0.98`, it has no idea *why* the policy regressed. Passing per-family speedups, cache hit rates, preemption counts, or sandbox violation tracebacks helps the agent diagnose what failed and formulate a targeted next hypothesis.

Always keep the unmodified `speedup`, `by_family`, and `valid` fields in the dictionary returned by `evaluate()` so your logs record both your search objective and the official grading metric.

In [ ]:
if r.valid:
    objectives = {
        "overall speedup": r.speedup,
        "document QA speedup": r.by_family["W3"],
        "worst-category speedup": min(r.by_family.values()),
    }
    print(objectives)
else:
    print("Exclude this candidate from selection:", r.violations)

## 4. Build and ablate your agentic research system (`research/`)

We provide a minimal 150-line sequential script in `examples/research_system.py` to illustrate the basic plumbing: it feeds the policy API, evaluator source, current best code, and a 1-line summary of recent scores into a single prompt, extracts a Python code block, scores it, and greedily keeps the highest-objective valid file.

**Do not just run `examples/research_system.py` as-is.** In Part 3, you should design and implement your own research system in `research/` (by substantially extending the starter script, adapting an open-source framework such as OpenEvolve, ShinkaEvolve, or CORAL, or building your own multi-agent or tree-search system). As you design your system, think carefully about what makes automated systems research succeed or fail:

- **How do agents form hypotheses and use diagnostics?** Does separating diagnosis/planning from code generation (or giving agents per-family cache hit rates, preemption counts, and queue wait times) produce more targeted improvements than one-shot code mutation?
- **How does your system handle invalid or crashing candidates?** Policies frequently fail due to syntax errors, sandbox rules, 50 ms hook timeouts, or evicting active blocks. Does your system feed the exact `violations` back to repair the candidate, or discard it blindly?
- **How does your system balance exploration and exploitation?** Greedy single-lineage mutation (`examples/research_system.py`) can easily get trapped in a local optimum or overfit to one family while breaking another. Maintaining multiple lineages, a search tree, or an archive of diverse policies can help combine mechanisms that work on different workloads.
- **How do you budget evaluations?** Scoring all 32 `public_dev` traces on every broken draft is slow; you might screen candidates on a fast subset of representative traces first and run full `public_dev` evaluation only on promising candidates.

First, run the cell below in `--offline` mode to see how the starter script logs candidates to `log.jsonl`, `calls.jsonl`, `campaign.json`, and `selection.json`. Then build your own system in `research/` and ensure it writes the same log records so you can audit and plot your campaigns.

In [ ]:
from datetime import datetime
USE_BEDROCK = False
EVALUATOR = Path("examples/evaluator.py")  # set to research/evaluator.py after adapting the example
DEMO_RUN = Path("runs") / f"example-demo-{datetime.now():%Y%m%d-%H%M%S-%f}"
OFFLINE = "" if USE_BEDROCK else "--offline"
!{sys.executable} examples/research_system.py --evaluator {EVALUATOR} --iters 3 {OFFLINE} --out {DEMO_RUN}

### Evaluate your research system: Ablation (Q2), Autonomy (Q3), and Cost (Q4)

As you run campaigns on `public_dev`, you should evaluate **what is working and what is not** in your research system itself:

- **System ablation study (Q2):** Hold the Bedrock model and token budget fixed, and remove, replace, or disable one or two components of your research system (for example: rich per-family diagnostic feedback vs. scalar `S` only; multi-lineage/tree search vs. greedy single-best; explicit hypothesis/repair loop vs. direct mutation; or your system vs. `examples/research_system.py`). Measure how the change affects valid-candidate rate and best speedup.
- **Autonomy and failure recovery (Q3):** Track your human interventions in an intervention log (`JSONL`) whenever you stop/restart a run or change prompts/code between campaigns. Inspect your campaign logs (`pXX.reply.txt`, `pXX.scores.json`) to find concrete examples where your agents diagnosed a failed or regressed candidate and proposed a useful follow-up—or where they got stuck.
- **Cost and diminishing returns (Q4):** Use `campaign_plots` below to plot best `dev_S` against both candidates scored and cumulative LLM tokens (including failed candidates, which still cost tokens), and identify when additional spending stopped producing meaningful gains.

In [ ]:
def campaign_plots(log_path):
    rows = [json.loads(line) for line in Path(log_path).read_text().splitlines() if line.strip()]
    best, best_curve, tokens, tok_curve = 0.0, [], 0, []
    for r in rows:
        tokens += r.get("tokens", 0)
        if r.get("valid") and r.get("dev_S"):
            best = max(best, r["dev_S"])
        best_curve.append(best)
        tok_curve.append(tokens)
    valid = sum(bool(r.get("valid")) for r in rows) / max(len(rows), 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    axes[0].plot(range(1, len(rows) + 1), best_curve)
    axes[0].set_xlabel("candidates scored")
    axes[1].plot(tok_curve, best_curve)
    axes[1].set_xlabel("LLM tokens")
    for ax in axes:
        ax.set_ylabel("best dev S so far")
        ax.axhline(1.0, color="gray", lw=1)
    fig.suptitle(f"{log_path}: {len(rows)} candidates, {valid:.0%} valid")
    plt.show()

campaign_plots(DEMO_RUN / "log.jsonl")        # then point it at your own campaigns

## 5. Select and validate finalists (`public_val`)

Use `public_dev` during search, and evaluate only your top finalists on `public_val` (16 traces) under the same two-replica configuration. Record `val_S` for each finalist and plot `dev_S` vs. `val_S` below. Points below the dashed diagonal have lower validation speedup than development speedup; check whether any drop is consistent across candidates or a sign that a policy overfit to `public_dev`.

In [ ]:
finalist_paths = [
    # e.g., "candidates/candidate_01.py", "candidates/candidate_02.py"
]
for path in finalist_paths:
    result = score(path, split="public_val")
    print(Path(path).name, result.valid, f"val_S={result.speedup:.4f}", result.by_family)

In [ ]:
finalist_scores = {
    # e.g., "candidate_01": (1.031, 1.012),  # (dev_S, val_S)
}
if finalist_scores:
    fig, ax = plt.subplots()
    for name, (dev_s, val_s) in finalist_scores.items():
        ax.scatter(dev_s, val_s)
        ax.annotate(name, (dev_s, val_s))
    ax.axline((1, 1), slope=1, color="gray", linestyle="--")
    ax.set(xlabel="Development speedup (dev_S)", ylabel="Validation speedup (val_S)")
    plt.show()

## 6. Understand and ablate the final policy (Q5 & Q6)

Do not treat your winning `policy.py` as a black box. You must be able to explain **what solution your agents came up with, why it works, and where it fails**:

1. **Read and explain the code (Q5):** Read your selected `policy.py` line by line and summarize its logic in your own words with clean pseudocode. What does it do in `on_event`, `target_concurrency`, `dispatch_order`, `route_call`, `admission_order`, `batch_token_budget`, `choose_preempted`, and `choose_victim`? Walk through a concrete trace (using `viz.record`) to show how its decisions reduce job latency compared with `policies/reference.py`.
2. **Verify the agents' hypothesis with a policy ablation (Q5):** LLM agents often include dead code, redundant heuristics, or plausible-sounding comments that do not actually explain the speedup. Disable one key mechanism in your final policy (while keeping everything else fixed) and measure the change in `S` and `by_family` below. Does the ablation confirm the agents' explanation?
3. **Diagnose failure modes (Q6):** Inspect `result.by_family` and `result.by_trace` to find a workload or trace where your policy underperforms or barely improves. Explain the system bottleneck on that case and predict how your policy will generalize to the hidden test set (which combines conditions such as bursty arrivals, tight memory, and mixed workloads).

In [ ]:
final_policy_path = "candidates/final_policy.py"
base_params = {}
# Replace this example with a parameter or modified copy that disables one mechanism in your final policy.
policy_ablations = {
    "no_adaptive_cap": {"adaptive_cap": False},
}
if Path(final_policy_path).exists():
    full_res = score(final_policy_path, split="public_val", params=base_params)
    for name, ablated_params in policy_ablations.items():
        merged = {**base_params, **ablated_params}
        ablated_res = score(final_policy_path, split="public_val", params=merged)
        change = " ".join(
            f"{f}:{ablated_res.by_family[f] / full_res.by_family[f]:.3f}"
            for f in full_res.by_family
        )
        print(f"{name:24} S={ablated_res.speedup:.4f} (full {full_res.speedup:.4f})  {change}")

## Questions

Answer Q1–Q6 below, keeping each question above its answer. Include your answers and supporting figures in `assignment4_report.pdf` after Part 1. Cite run IDs and candidate hashes so every claim can be verified against your campaign logs.

**Q1. How does your agentic research system work?** Include a diagram showing its LLM calls, deterministic steps, tools, stored state, and points where a person can intervene. Explain your evaluator's objective, the feedback given to agents, how invalid candidates are handled, and how the system selects a final policy and decides when to stop. Explain why you chose this architecture.

**Your answer:**

**Q2. What did your ablation study show?** Ablate one or two components or design choices in your research system while holding the model and token budget fixed. Describe what you removed, replaced, or disabled, show the results, and explain what the comparison reveals. For example, remove evaluation feedback, disable shared memory, or replace a candidate-selection strategy. The supplied sequential system can serve as one comparison.

**Your answer:**

**Q3. How autonomous was it in practice?** Describe the longest unattended campaign and summarize the intervention log. Show one case where an agent diagnosed a failure and chose a useful follow-up experiment, or explain why it could not.

**Your answer:**

**Q4. Did more spending help?** Plot the best development speedup against tokens used and against candidates evaluated. Mark your finalists' validation scores, report the fraction of candidates that were valid, and explain where you would stop spending.

**Your answer:**

**Q5. What policy did the agents find?** Explain the policy in your own words, with pseudocode and a short trace example. Why should its decisions reduce mean job latency? Disable one discovered mechanism, measure the effect, and use the results to assess the agents' explanation.

**Your answer:**

**Q6. Where does it fail, and what does that say about agents doing systems research?** Report results for each workload category and examine one case where the policy performs poorly. Predict how it will perform on the hidden test set. What do your experiments show about the strengths and limitations of using agents for this research?

**Your answer:**

## Submission checklist

Before submitting, verify that your Gradescope and Canvas submissions include:

- **Policy package (Gradescope & Canvas):** `policy.py`, optional `params.json`, and `submission.json` (recording `r.policy_sha256`, Python version, and any non-stdlib imports).
- **Campaign materials (Canvas):** `research/` (system and evaluator source, configuration, and prompts), `candidates/` (all generated policies, including failed ones), and `runs/` (campaign logs, token receipts, and human intervention logs).
- **Report (Canvas):** `assignment4_report.pdf` containing Part 1 Q1–Q3 and Part 3 Q1–Q6 in order (with each original question above its answer), required tables/plots, run IDs, candidate hashes, `pytest` summary, and token usage totals. See [Submission](../README.md#submitting) for the exact archive layout.